# Fine-tuning Bodhan Indic-Transcribe (ASR) on conversational Marathi

Colab notebook (A100 80GB) that produced the results in the [GitHub repo](https://github.com/deveshh01/bodhan-asr-marathi). The repository is the source of truth for the code: the `%%writefile` cells below hold its **first** version; later fixes were pulled into the VM with `git clone` (section 7). `scripts/run_pipeline.sh` reproduces everything in one resumable command.

Debugging cells used during development were removed; outputs are from the actual runs.

## 0. Environment + gated model access

In [2]:
# Environment check: GPU + HF token (stored as a Colab secret named HF_TOKEN)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1; free -g | head -2
from google.colab import userdata
try:
    tok = userdata.get('HF_TOKEN'); print('HF_TOKEN secret found:', bool(tok))
except Exception as e:
    print('HF_TOKEN secret missing ->', type(e).__name__)

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
overlay         236G   48G  189G  21% /
               total        used        free      shared  buff/cache   available
Mem:             167           3         160           0           3         163
HF_TOKEN secret found: True


In [3]:
# Download the gated model snapshot (HF weights only; .nemo skipped) and inspect its remote code
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download('bodhan-ai/indic-transcribe-core', ignore_patterns=['nemo/*.nemo', '*.png'], local_dir='/content/models/indic-transcribe-core')
!ls -la {MODEL_DIR}; cat {MODEL_DIR}/requirements.txt {MODEL_DIR}/config.json {MODEL_DIR}/generation_config.json {MODEL_DIR}/tokenizer_config.json

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

total 4776760
drwxr-xr-x 4 root root       4096 Sep 23 15:31 .
drwxr-xr-x 3 root root       4096 Sep 23 15:31 ..
-rw-r--r-- 1 root root      25546 Sep 23 15:31 Bodhan_AI_Open_Model_License.md
drwxr-xr-x 3 root root       4096 Sep 23 15:31 .cache
-rw-r--r-- 1 root root        923 Sep 23 15:31 config.json
-rw-r--r-- 1 root root       2162 Sep 23 15:31 configuration_indic_canary.py
-rw-r--r-- 1 root root       5998 Sep 23 15:31 feature_extraction_indic_canary.py
-rw-r--r-- 1 root root     133368 Sep 23 15:31 feature_extractor.safetensors
-rw-r--r-- 1 root root        175 Sep 23 15:31 generation_config.json
-rw-r--r-- 1 root root       1634 Sep 23 15:31 .gitattributes
-rw-r--r-- 1 root root       3565 Sep 23 15:31 indic-open-license.md
-rw-r--r-- 1 root root      12849 Sep 23 15:31 indic_transcribe.py
-rw-r--r-- 1 root root       4966 Sep 23 15:31 inference.py
-rw-r--r-- 1 root root      10534 Sep 23 15:31 lid.py
-rw-r--r-- 1 root root       8222 Sep 23 15:31 long_form.py
-rw-r--r-- 1 root

## 1. Data: SPRING-INX Marathi R2 (10/47 train shards, validation shard 0, full test)

In [5]:
# 1. Install dependencies + download SPRING-INX Marathi R2 shards (IIT Madras, conversational Marathi, CC-BY-4.0)
!pip -q install jiwer peft soundfile librosa pyyaml sentencepiece 2>&1 | tail -1
import os
os.makedirs('/content/data/raw', exist_ok=True)
from huggingface_hub import hf_hub_download
REPO = 'SPRINGLab/SPRING_INX_Marathi_R2'
shards = [f'data/train-{i:05d}-of-00047.parquet' for i in range(10)] \
       + ['data/validation-00000-of-00006.parquet'] \
       + [f'data/test-{i:05d}-of-00002.parquet' for i in range(2)]
for s in shards:
    hf_hub_download(REPO, s, repo_type='dataset', local_dir='/content/data/raw')
!du -sh /content/data/raw/data; ls /content/data/raw/data | wc -l

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.6 MB/s eta 0:00:00


data/train-00000-of-00047.parquet: reconstructing file:   0%|          |  0.00B /  497MB            

data/train-00000-of-00047.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00047.parquet: reconstructing file:   0%|          |  0.00B /  504MB            

data/train-00001-of-00047.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  270MB            

data/test-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  263MB            

data/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

5.5G	/content/data/raw/data
13


## 2. Project code (first version; see the repo for the final code)

In [6]:
# 2. Project layout (source files below are mirrored 1:1 in the GitHub repo)
!mkdir -p /content/bodhan-asr-marathi/{src/bodhan_asr,scripts,configs}

In [7]:
%%writefile /content/bodhan-asr-marathi/src/bodhan_asr/__init__.py


Writing /content/bodhan-asr-marathi/src/bodhan_asr/__init__.py


In [8]:
%%writefile /content/bodhan-asr-marathi/src/bodhan_asr/text.py
"""Text normalisation for Marathi transcripts.

Two levels are used on purpose:

* ``clean_target`` - light cleaning applied to *training targets*. It keeps
  punctuation and casing because Indic-Transcribe is run with ``pnc="yes"`` and
  we don't want to teach it to drop punctuation.
* ``normalize_for_wer`` - aggressive normalisation applied to *both* reference and
  hypothesis only at scoring time, so WER/CER measure word choice, not
  punctuation/spacing conventions (standard practice for Indic ASR benchmarks).
"""

from __future__ import annotations

import re
import unicodedata

# Zero-width chars that show up in scraped/annotated Devanagari text.
# ZWJ/ZWNJ (U+200D/U+200C) can change conjunct rendering but not the word, so
# they're dropped for scoring; they're kept in targets.
_ZERO_WIDTH = dict.fromkeys(map(ord, "​‌‍⁠﻿"), None)

# Annotation tags such as <noise>, [laugh], (inaudible) used by some corpora.
_TAG_RE = re.compile(r"<[^>]*>|\[[^\]]*\]|\{[^}]*\}")
_WS_RE = re.compile(r"\s+")

# Everything that is punctuation/symbol in Unicode, plus the Devanagari danda(s).
_PUNCT_CHARS = "".join(
    chr(c)
    for c in range(0x110000)
    if unicodedata.category(chr(c)).startswith(("P", "S"))
) + "।॥"
_PUNCT_TABLE = dict.fromkeys(map(ord, _PUNCT_CHARS), " ")


def clean_target(text: str) -> str:
    """Light cleanup for training targets (keeps punctuation)."""
    text = unicodedata.normalize("NFC", text)
    text = _TAG_RE.sub(" ", text)
    return _WS_RE.sub(" ", text).strip()


def normalize_for_wer(text: str) -> str:
    """Aggressive normalisation used only for WER/CER scoring."""
    text = unicodedata.normalize("NFC", text).translate(_ZERO_WIDTH)
    text = _TAG_RE.sub(" ", text)
    text = text.translate(_PUNCT_TABLE).lower()  # lower() only affects code-mixed Latin
    return _WS_RE.sub(" ", text).strip()


_DEVANAGARI_RE = re.compile(r"[ऀ-ॿ]")
_LATIN_RE = re.compile(r"[A-Za-z]")


def script_stats(text: str) -> dict[str, int]:
    """Counts of Devanagari vs Latin letters - used to profile code-mixing."""
    return {
        "devanagari": len(_DEVANAGARI_RE.findall(text)),
        "latin": len(_LATIN_RE.findall(text)),
    }


Writing /content/bodhan-asr-marathi/src/bodhan_asr/text.py


In [9]:
%%writefile /content/bodhan-asr-marathi/src/bodhan_asr/metrics.py
"""WER / CER computation with the normalisation from :mod:`bodhan_asr.text`."""

from __future__ import annotations

import jiwer

from .text import normalize_for_wer


def compute_metrics(refs: list[str], hyps: list[str]) -> dict[str, float]:
    """Corpus-level WER and CER (percent) on normalised text.

    Empty references are dropped (WER is undefined for them); empty hypotheses
    are kept - they are genuine deletions.
    """
    pairs = [(normalize_for_wer(r), normalize_for_wer(h)) for r, h in zip(refs, hyps)]
    pairs = [(r, h) for r, h in pairs if r]
    if not pairs:
        return {"wer": float("nan"), "cer": float("nan"), "n": 0}
    r, h = map(list, zip(*pairs))
    return {
        "wer": 100 * jiwer.wer(r, h),
        "cer": 100 * jiwer.cer(r, h),
        "n": len(r),
    }


Writing /content/bodhan-asr-marathi/src/bodhan_asr/metrics.py


In [10]:
%%writefile /content/bodhan-asr-marathi/src/bodhan_asr/data.py
"""Dataset preparation and loading.

Raw SPRING-INX parquet shards are converted once into 16 kHz mono FLAC files plus
JSON-lines manifests in the NeMo format::

    {"audio_filepath": ..., "duration": ..., "text": ..., "source_lang": "mr",
     "target_lang": "mr", "pnc": "yes", "utt_id": ...}

Using NeMo-style manifests keeps the prepared data usable by both the
transformers training loop in this repo and NeMo's ``speech_to_text_aed.py``.
"""

from __future__ import annotations

import io
import json
import logging
import random
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import pyarrow.parquet as pq
import soundfile as sf
import torch
from torch.utils.data import Dataset

from .text import clean_target

log = logging.getLogger(__name__)
SAMPLE_RATE = 16_000


@dataclass
class FilterConfig:
    min_duration: float = 0.5
    max_duration: float = 30.0  # Canary's positional limit is 40 s; keep headroom.
    max_chars_per_sec: float = 25.0  # catches mis-segmented/misaligned utterances
    min_chars: int = 2


def _decode(audio_bytes: bytes) -> np.ndarray:
    wav, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)  # to mono
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    return wav


def prepare_split(
    shards: list[Path],
    out_dir: Path,
    split: str,
    filt: FilterConfig,
    lang: str = "mr",
    max_items: int | None = None,
    seed: int = 0,
) -> Path:
    """Decode parquet shards -> FLAC + manifest. Returns the manifest path."""
    audio_dir = out_dir / "audio" / split
    audio_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for shard in shards:
        table = pq.read_table(shard, columns=["utterance_id", "text", "audio"])
        rows.extend(table.to_pylist())
    if max_items is not None and len(rows) > max_items:
        rows = random.Random(seed).sample(rows, max_items)

    reasons: Counter[str] = Counter()
    seen: set[str] = set()
    manifest = out_dir / f"{split}.jsonl"
    total_sec = 0.0
    with manifest.open("w", encoding="utf-8") as f:
        for row in rows:
            uid = row["utterance_id"]
            if uid in seen:
                reasons["duplicate_id"] += 1
                continue
            seen.add(uid)
            text = clean_target(row["text"] or "")
            if len(text) < filt.min_chars:
                reasons["empty_text"] += 1
                continue
            try:
                wav = _decode(row["audio"]["bytes"])
            except Exception:  # corrupt audio in the source corpus
                reasons["decode_error"] += 1
                continue
            dur = len(wav) / SAMPLE_RATE
            if dur < filt.min_duration:
                reasons["too_short"] += 1
                continue
            if dur > filt.max_duration:
                reasons["too_long"] += 1
                continue
            if len(text) / dur > filt.max_chars_per_sec:
                reasons["chars_per_sec"] += 1
                continue
            path = audio_dir / f"{uid}.flac"
            sf.write(path, wav, SAMPLE_RATE)
            total_sec += dur
            reasons["kept"] += 1
            f.write(json.dumps({
                "audio_filepath": str(path), "duration": round(dur, 3), "text": text,
                "source_lang": lang, "target_lang": lang, "pnc": "yes", "utt_id": uid,
            }, ensure_ascii=False) + "\n")
    log.info("%s: %s | %.2f h kept", split, dict(reasons), total_sec / 3600)
    return manifest


def read_manifest(path: str | Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


class ManifestDataset(Dataset):
    """Returns ``{"audio": float32 np.ndarray @16k, "text": str, "utt_id": str}``."""

    def __init__(self, manifest: str | Path, max_items: int | None = None):
        self.items = read_manifest(manifest)
        if max_items is not None:
            self.items = self.items[:max_items]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, i: int) -> dict:
        it = self.items[i]
        wav, _ = sf.read(it["audio_filepath"], dtype="float32")
        return {"audio": wav, "text": it["text"], "utt_id": it["utt_id"],
                "duration": it["duration"]}


class DurationBucketSampler(torch.utils.data.Sampler[list[int]]):
    """Batches utterances of similar length up to ``max_batch_seconds`` of audio.

    Cuts padding waste substantially vs. random batching (conversational speech
    ranges from <1 s to 30 s), which matters for a 1.2B-param encoder.
    """

    def __init__(self, durations: list[float], max_batch_seconds: float,
                 max_batch_size: int = 64, shuffle: bool = True, seed: int = 0):
        self.durations = durations
        self.max_batch_seconds = max_batch_seconds
        self.max_batch_size = max_batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self._batches = self._make_batches()

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch
        self._batches = self._make_batches()

    def _make_batches(self) -> list[list[int]]:
        rng = random.Random(self.seed + self.epoch)
        idx = sorted(range(len(self.durations)), key=lambda i: self.durations[i])
        batches, cur, cur_max = [], [], 0.0
        for i in idx:
            d = self.durations[i]
            # padded cost of the batch = batch_size * longest item
            if cur and (max(cur_max, d) * (len(cur) + 1) > self.max_batch_seconds
                        or len(cur) >= self.max_batch_size):
                batches.append(cur)
                cur, cur_max = [], 0.0
            cur.append(i)
            cur_max = max(cur_max, d)
        if cur:
            batches.append(cur)
        if self.shuffle:
            rng.shuffle(batches)
        return batches

    def __iter__(self):
        return iter(self._batches)

    def __len__(self) -> int:
        return len(self._batches)


Writing /content/bodhan-asr-marathi/src/bodhan_asr/data.py


In [11]:
%%writefile /content/bodhan-asr-marathi/src/bodhan_asr/config.py
"""Typed experiment config loaded from YAML (with CLI ``key=value`` overrides)."""

from __future__ import annotations

import dataclasses
from dataclasses import dataclass, field
from pathlib import Path

import yaml


@dataclass
class DataConfig:
    train_manifest: str = "/content/data/prepared/train.jsonl"
    dev_manifest: str = "/content/data/prepared/dev.jsonl"
    test_manifest: str = "/content/data/prepared/test.jsonl"
    lang: str = "mr"
    max_batch_seconds: float = 600.0  # padded audio-seconds per micro-batch
    max_batch_size: int = 48
    num_workers: int = 8
    dev_max_items: int | None = None


@dataclass
class ModelConfig:
    name_or_path: str = "bodhan-ai/indic-transcribe-core"
    # full | decoder_only | lora
    strategy: str = "full"
    freeze_encoder_layers: int = 0  # freeze the bottom N FastConformer layers
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_target_modules: list[str] | None = None  # None -> all attention projections
    # canary2 output mode used in the prompt: native | mixed | romanised
    prompt_mode: str = "mixed"
    spec_augment: bool = True
    label_smoothing: float = 0.0


@dataclass
class TrainConfig:
    output_dir: str = "/content/runs/mr_full"
    seed: int = 42
    lr: float = 1e-5
    encoder_lr_scale: float = 1.0  # encoder LR = lr * scale (discriminative LR)
    weight_decay: float = 0.01
    warmup_steps: int = 200
    max_steps: int = 3000
    grad_accum: int = 1
    max_grad_norm: float = 1.0
    precision: str = "bf16"  # bf16 | fp16 | fp32
    eval_every: int = 250
    log_every: int = 10
    save_best: bool = True
    early_stopping_patience: int = 5  # in evals; 0 disables
    num_beams: int = 1
    max_new_tokens: int = 256


@dataclass
class ExperimentConfig:
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

    @classmethod
    def load(cls, path: str | Path, overrides: list[str] | None = None) -> "ExperimentConfig":
        raw = yaml.safe_load(Path(path).read_text()) or {}
        for ov in overrides or []:
            key, val = ov.split("=", 1)
            section, name = key.split(".", 1)
            raw.setdefault(section, {})[name] = yaml.safe_load(val)
        return cls(
            data=DataConfig(**raw.get("data", {})),
            model=ModelConfig(**raw.get("model", {})),
            train=TrainConfig(**raw.get("train", {})),
        )

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)


Writing /content/bodhan-asr-marathi/src/bodhan_asr/config.py


In [12]:
%%writefile /content/bodhan-asr-marathi/scripts/prepare_data.py
"""Download (optional) and prepare SPRING-INX Marathi R2 into FLAC + manifests.

Example::

    python scripts/prepare_data.py --raw_dir /content/data/raw --out_dir /content/data/prepared \
        --train_shards 10 --dev_items 1000
"""

from __future__ import annotations

import argparse
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from bodhan_asr.data import FilterConfig, prepare_split  # noqa: E402

REPO = "SPRINGLab/SPRING_INX_Marathi_R2"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw_dir", type=Path, required=True)
    ap.add_argument("--out_dir", type=Path, required=True)
    ap.add_argument("--train_shards", type=int, default=10, help="of 47 available")
    ap.add_argument("--dev_items", type=int, default=1000,
                    help="random subset of validation shard 0 used for model selection")
    ap.add_argument("--download", action="store_true")
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

    names = {
        "train": [f"data/train-{i:05d}-of-00047.parquet" for i in range(args.train_shards)],
        "dev": ["data/validation-00000-of-00006.parquet"],
        "test": [f"data/test-{i:05d}-of-00002.parquet" for i in range(2)],
    }
    if args.download:
        from huggingface_hub import hf_hub_download
        for files in names.values():
            for n in files:
                hf_hub_download(REPO, n, repo_type="dataset", local_dir=args.raw_dir)

    filt = FilterConfig()
    for split, files in names.items():
        shards = [args.raw_dir / n for n in files]
        # The official test split is never filtered by duration/text heuristics
        # beyond decodability, so reported test numbers stay comparable.
        split_filt = FilterConfig(max_duration=40.0, max_chars_per_sec=1e9) if split == "test" else filt
        prepare_split(shards, args.out_dir, split, split_filt,
                      max_items=args.dev_items if split == "dev" else None)


if __name__ == "__main__":
    main()


Writing /content/bodhan-asr-marathi/scripts/prepare_data.py


In [13]:
%%writefile /content/bodhan-asr-marathi/src/bodhan_asr/model.py
"""Training adapter around Bodhan's Indic-Transcribe (IndicCanary) HF port.

The released transformers port is *inference-only*: ``forward(labels=...)``
raises ``NotImplementedError``. This module adds what training needs without
modifying the upstream code:

* teacher-forced cross-entropy over the target tokens (the fixed 10-token
  canary2 prompt is masked out of the loss, as in NeMo's prompt-masked loss);
* SpecAugment on the log-mel features (the port has no dropout at all, so
  this is the main regulariser);
* BatchNorm running statistics frozen - the FastConformer conv module's BN
  would otherwise re-estimate stats from small, padding-contaminated batches;
* three fine-tuning strategies: ``full``, ``decoder_only`` and ``lora``.

Everything the model sees is built from the upstream tokenizer / feature
extractor so train-time and inference-time inputs are identical.
"""

from __future__ import annotations

import logging
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import ExperimentConfig

log = logging.getLogger(__name__)

# Files that make a checkpoint directory self-contained (loadable with the
# upstream ``IndicTranscribe.from_pretrained``).
_ASSET_GLOBS = ["*.py", "tokenizer_*.model", "tokenizer_config.json",
                "feature_extractor.safetensors", "generation_config.json", "*.md"]

# Module names inside IndicCanary attention blocks (encoder / decoder).
LORA_TARGETS = ["linear_q", "linear_k", "linear_v", "linear_out",
                "query_net", "key_net", "value_net", "out_projection"]


def _resolve_model_dir(name_or_path: str) -> Path:
    p = Path(name_or_path)
    if p.exists():
        return p
    from huggingface_hub import snapshot_download
    return Path(snapshot_download(name_or_path, ignore_patterns=["nemo/*.nemo", "*.png"]))


def _import_upstream(model_dir: Path):
    """Import the upstream remote-code modules as flat modules (they support it)."""
    if str(model_dir) not in sys.path:
        sys.path.insert(0, str(model_dir))
    from feature_extraction_indic_canary import IndicCanaryFeatureExtractor
    from modeling_indic_canary import IndicCanaryForConditionalGeneration
    from tokenization_indic_canary import IndicCanaryTokenizer
    return IndicCanaryForConditionalGeneration, IndicCanaryFeatureExtractor, IndicCanaryTokenizer


class SpecAugment(nn.Module):
    """Frequency + time masking on (B, n_mels, T) features (NeMo Canary defaults)."""

    def __init__(self, freq_masks=2, freq_width=27, time_masks=10, time_width=0.05):
        super().__init__()
        self.freq_masks, self.freq_width = freq_masks, freq_width
        self.time_masks, self.time_width = time_masks, time_width

    @torch.no_grad()
    def forward(self, feats: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        b, n_mels, _ = feats.shape
        feats = feats.clone()
        for i in range(b):
            t = int(lengths[i])
            for _ in range(self.freq_masks):
                w = np.random.randint(0, self.freq_width + 1)
                f0 = np.random.randint(0, max(1, n_mels - w))
                feats[i, f0:f0 + w, :] = 0.0
            max_w = max(1, int(self.time_width * t))
            for _ in range(self.time_masks):
                w = np.random.randint(0, max_w + 1)
                t0 = np.random.randint(0, max(1, t - w))
                feats[i, :, t0:t0 + w] = 0.0
        return feats


class AsrAdapter:
    """Owns model + tokenizer + feature extractor and exposes the train/eval API
    used by :mod:`bodhan_asr.trainer`."""

    def __init__(self, model, fe, tokenizer, cfg: ExperimentConfig, model_dir: Path,
                 device: str = "cuda"):
        self.model, self.fe, self.tok, self.cfg = model, fe, tokenizer, cfg
        self.model_dir, self.device = model_dir, device
        self.lang = cfg.data.lang
        # "mixed" (itn) mode: SPRING-INX writes English loanwords in Latin script,
        # which is exactly what Indic-Transcribe's <|itn|> slot produces.
        self.prompt = tokenizer.encode_prompt(self.lang, itn=cfg.model.prompt_mode == "mixed",
                                              romanized=cfg.model.prompt_mode == "romanised")
        self.spec_augment = SpecAugment() if cfg.model.spec_augment else None

    # ------------------------------------------------------------------ setup
    @classmethod
    def from_config(cls, cfg: ExperimentConfig, checkpoint: str | None = None) -> "AsrAdapter":
        model_dir = _resolve_model_dir(cfg.model.name_or_path)
        Model, FE, Tok = _import_upstream(model_dir)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        strategy = cfg.model.strategy
        weights_dir = checkpoint if (checkpoint and strategy != "lora") else model_dir
        # fp32 master weights; bf16 compute comes from autocast in the trainer.
        model = Model.from_pretrained(str(weights_dir), dtype=torch.float32)
        adapter = cls(model, FE.from_pretrained(str(model_dir), device=device),
                      Tok.from_pretrained(str(model_dir)), cfg, model_dir, device)
        adapter._apply_strategy()
        if checkpoint and strategy == "lora":
            from peft import set_peft_model_state_dict
            from safetensors.torch import load_file
            set_peft_model_state_dict(model, load_file(str(Path(checkpoint) / "adapter.safetensors")))
            log.info("loaded LoRA adapter from %s", checkpoint)
        model.to(device)
        n_train = sum(p.numel() for p in adapter.trainable_parameters())
        n_all = sum(p.numel() for p in model.parameters())
        log.info("strategy=%s trainable params %.1fM / %.1fM (%.1f%%)", strategy,
                 n_train / 1e6, n_all / 1e6, 100 * n_train / n_all)
        return adapter

    def _apply_strategy(self) -> None:
        mc, enc = self.cfg.model, self.model.model.encoder
        if mc.strategy == "lora":
            from peft import LoraConfig, inject_adapter_in_model
            for p in self.model.parameters():
                p.requires_grad_(False)
            # inject (not get_peft_model) so the upstream generate()/forward
            # signatures stay untouched.
            inject_adapter_in_model(LoraConfig(
                r=mc.lora_r, lora_alpha=mc.lora_alpha, lora_dropout=mc.lora_dropout,
                target_modules=mc.lora_target_modules or LORA_TARGETS), self.model)
        elif mc.strategy == "decoder_only":
            for p in enc.parameters():
                p.requires_grad_(False)
        elif mc.strategy != "full":
            raise ValueError(f"unknown strategy {mc.strategy!r}")
        # Optionally freeze the subsampling + bottom N conformer layers (acoustic
        # front-end generalises well; saves memory and limits forgetting).
        if mc.freeze_encoder_layers:
            for p in enc.pre_encode.parameters():
                p.requires_grad_(False)
            for layer in enc.layers[: mc.freeze_encoder_layers]:
                for p in layer.parameters():
                    p.requires_grad_(False)

    def trainable_parameters(self):
        return [p for p in self.model.parameters() if p.requires_grad]

    def param_groups(self, lr: float, encoder_lr_scale: float, weight_decay: float):
        enc_ids = {id(p) for p in self.model.model.encoder.parameters()}
        groups = {("enc", True): [], ("enc", False): [], ("dec", True): [], ("dec", False): []}
        for name, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            decay = p.ndim >= 2 and "embedding" not in name  # no WD on norms/biases/embeddings
            groups[("enc" if id(p) in enc_ids else "dec", decay)].append(p)
        out = []
        for (part, decay), params in groups.items():
            if params:
                out.append({"params": params, "weight_decay": weight_decay if decay else 0.0,
                            "lr": lr * (encoder_lr_scale if part == "enc" else 1.0)})
        return out

    # ------------------------------------------------------------------- data
    def encode_text(self, text: str) -> list[int]:
        """Target token ids: multilingual SPM pieces offset into the aggregate vocab."""
        return [i + self.tok.spl_size for i in self.tok.multi.encode(text)]

    def collate(self, items: list[dict]) -> dict:
        min_len = self.fe.sample_rate  # upstream pads clips < 1 s to 1 s, centred
        lens = [max(len(it["audio"]), min_len) for it in items]
        audio = torch.zeros(len(items), max(lens))
        for i, it in enumerate(items):
            wav = torch.from_numpy(np.asarray(it["audio"], dtype=np.float32))
            off = (min_len - len(wav)) // 2 if len(wav) < min_len else 0
            audio[i, off:off + len(wav)] = wav
        max_tok = self.model.config.max_target_positions - len(self.prompt) - 1
        seqs = [self.prompt + self.encode_text(it["text"])[:max_tok] + [self.tok.eos_id] for it in items]
        tokens = torch.full((len(seqs), max(map(len, seqs))), self.tok.pad_id, dtype=torch.long)
        for i, s in enumerate(seqs):
            tokens[i, : len(s)] = torch.tensor(s)
        return {"audio": audio, "audio_lens": torch.tensor(lens), "tokens": tokens,
                "token_lens": torch.tensor([len(s) for s in seqs]),
                "texts": [it["text"] for it in items], "utt_ids": [it["utt_id"] for it in items]}

    def to_device(self, batch: dict) -> dict:
        batch = dict(batch)
        audio = batch["audio"].to(self.device, non_blocking=True)
        feats, feat_lens = self.fe(audio, batch["audio_lens"].to(self.device))
        batch["feats"], batch["feat_lens"] = feats, feat_lens
        batch["feat_mask"] = (torch.arange(feats.size(2), device=self.device)[None]
                              < feat_lens[:, None]).long()
        batch["tokens"] = batch["tokens"].to(self.device, non_blocking=True)
        batch["token_lens"] = batch["token_lens"].to(self.device)
        return batch

    # ------------------------------------------------------------- train/eval
    def train_mode(self) -> None:
        self.model.train()
        for m in self.model.modules():  # freeze BN running stats (see module doc)
            if isinstance(m, nn.BatchNorm1d):
                m.eval()

    def loss(self, batch: dict) -> torch.Tensor:
        feats = batch["feats"]
        if self.model.training and self.spec_augment is not None:
            feats = self.spec_augment(feats, batch["feat_lens"])
        tokens = batch["tokens"]
        dec_in, target = tokens[:, :-1], tokens[:, 1:].clone()
        # target position j predicts tokens[j+1]; the first len(prompt)-1 targets
        # are prompt tokens -> masked. Padding after eos is masked too.
        pos = torch.arange(target.size(1), device=target.device)[None]
        target[(pos < len(self.prompt) - 1) | (pos >= (batch["token_lens"][:, None] - 1))] = -100
        out = self.model(input_features=feats, attention_mask=batch["feat_mask"],
                         decoder_input_ids=dec_in, use_cache=False)
        return F.cross_entropy(out.logits.float().transpose(1, 2), target, ignore_index=-100,
                               label_smoothing=self.cfg.model.label_smoothing)

    @torch.no_grad()
    def transcribe(self, batch: dict, num_beams: int = 1, max_new_tokens: int = 256) -> list[str]:
        prompt = torch.tensor([self.prompt] * batch["feats"].size(0), device=self.device)
        out = self.model.generate(input_features=batch["feats"], attention_mask=batch["feat_mask"],
                                  decoder_input_ids=prompt, max_new_tokens=max_new_tokens,
                                  num_beams=num_beams, do_sample=False)
        texts = []
        for row in out.tolist():
            ids = self.tok.strip_prompt_and_trim(row, self.prompt)
            if self.tok.eos_id in ids:  # anything after the first eos is padding
                ids = ids[: ids.index(self.tok.eos_id)]
            texts.append(self.tok.decode(ids))
        return texts

    # ------------------------------------------------------------------- save
    def save(self, out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        if self.cfg.model.strategy == "lora":
            from peft import get_peft_model_state_dict
            from safetensors.torch import save_file
            sd = {k: v.detach().cpu().contiguous() for k, v in get_peft_model_state_dict(self.model).items()}
            save_file(sd, str(out_dir / "adapter.safetensors"))
        else:
            # bf16 *copy* on disk (2.4 GB vs 4.9 GB; inference runs in bf16 anyway).
            # Never cast the live model: that would truncate the fp32 master
            # weights the optimizer is still updating. lm_head is tied to the
            # token embedding and re-tied on load, so it isn't stored twice.
            sd = {k: v.detach().to(torch.bfloat16) for k, v in self.model.state_dict().items()
                  if k != "lm_head.weight"}
            self.model.save_pretrained(str(out_dir), state_dict=sd, safe_serialization=True)
            for pat in _ASSET_GLOBS:  # make the dir loadable by upstream IndicTranscribe
                for f in self.model_dir.glob(pat):
                    shutil.copy2(f, out_dir / f.name)
        log.info("saved %s checkpoint -> %s", self.cfg.model.strategy, out_dir)


Writing /content/bodhan-asr-marathi/src/bodhan_asr/model.py


In [14]:
%%writefile /content/bodhan-asr-marathi/src/bodhan_asr/trainer.py
"""Minimal, explicit fine-tuning loop.

Written by hand (rather than ``Seq2SeqTrainer``) because Indic-Transcribe is a
``trust_remote_code`` model with a custom prompt format and feature extractor:
owning the loop makes it obvious exactly what the model sees and lets us use a
duration-bucketed sampler and WER-based model selection.
"""

from __future__ import annotations

import json
import logging
import math
import time
from contextlib import nullcontext
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from .config import ExperimentConfig
from .data import DurationBucketSampler, ManifestDataset
from .metrics import compute_metrics
from .model import AsrAdapter

log = logging.getLogger(__name__)


def _autocast(precision: str):
    if precision == "bf16":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    if precision == "fp16":
        return torch.autocast("cuda", dtype=torch.float16)
    return nullcontext()


def _lr_lambda(warmup: int, total: int):
    """Linear warmup then cosine decay to 10% of peak."""
    def f(step: int) -> float:
        if step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(1, total - warmup))
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
    return f


def make_loader(ds: ManifestDataset, adapter: AsrAdapter, cfg: ExperimentConfig,
                shuffle: bool) -> DataLoader:
    sampler = DurationBucketSampler(
        [it["duration"] for it in ds.items],
        max_batch_seconds=cfg.data.max_batch_seconds,
        max_batch_size=cfg.data.max_batch_size,
        shuffle=shuffle, seed=cfg.train.seed,
    )
    return DataLoader(ds, batch_sampler=sampler, collate_fn=adapter.collate,
                      num_workers=cfg.data.num_workers, pin_memory=True,
                      persistent_workers=cfg.data.num_workers > 0)


@torch.no_grad()
def evaluate(adapter: AsrAdapter, loader: DataLoader, cfg: ExperimentConfig,
             predictions_path: Path | None = None) -> dict:
    adapter.model.eval()
    refs, hyps, ids, loss_sum, n_batches = [], [], [], 0.0, 0
    for batch in loader:
        batch = adapter.to_device(batch)
        with _autocast(cfg.train.precision):
            loss_sum += adapter.loss(batch).item()
            n_batches += 1
            hyps += adapter.transcribe(batch, num_beams=cfg.train.num_beams,
                                       max_new_tokens=cfg.train.max_new_tokens)
        refs += batch["texts"]
        ids += batch["utt_ids"]
    metrics = compute_metrics(refs, hyps)
    metrics["loss"] = loss_sum / max(1, n_batches)
    if predictions_path is not None:
        predictions_path.parent.mkdir(parents=True, exist_ok=True)
        with predictions_path.open("w", encoding="utf-8") as f:
            for i, r, h in zip(ids, refs, hyps):
                f.write(json.dumps({"utt_id": i, "ref": r, "hyp": h}, ensure_ascii=False) + "\n")
    adapter.train_mode()
    return metrics


def train(cfg: ExperimentConfig) -> dict:
    out = Path(cfg.train.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "config.json").write_text(json.dumps(cfg.to_dict(), indent=2))
    torch.manual_seed(cfg.train.seed)

    adapter = AsrAdapter.from_config(cfg)
    train_ds = ManifestDataset(cfg.data.train_manifest)
    dev_ds = ManifestDataset(cfg.data.dev_manifest, max_items=cfg.data.dev_max_items)
    train_loader = make_loader(train_ds, adapter, cfg, shuffle=True)
    dev_loader = make_loader(dev_ds, adapter, cfg, shuffle=False)
    log.info("train utts=%d (%.1f h) batches/epoch=%d | dev utts=%d", len(train_ds),
             sum(it["duration"] for it in train_ds.items) / 3600, len(train_loader), len(dev_ds))

    groups = adapter.param_groups(cfg.train.lr, cfg.train.encoder_lr_scale, cfg.train.weight_decay)
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8, fused=True)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(cfg.train.warmup_steps, cfg.train.max_steps))
    scaler = torch.amp.GradScaler(enabled=cfg.train.precision == "fp16")
    tb = SummaryWriter(out / "tb")
    history = out / "history.jsonl"

    def log_row(row: dict) -> None:
        with history.open("a") as f:
            f.write(json.dumps(row) + "\n")
        for k, v in row.items():
            if isinstance(v, (int, float)) and k != "step":
                tb.add_scalar(k, v, row["step"])

    # Step-0 evaluation = zero-shot baseline on the same dev set.
    best = evaluate(adapter, dev_loader, cfg)
    log.info("step 0 (zero-shot) dev: %s", best)
    log_row({"step": 0, **{f"dev/{k}": v for k, v in best.items()}})
    best_wer, bad_evals = best["wer"], 0

    step, epoch, t0 = 0, 0, time.time()
    adapter.train_mode()
    running = 0.0
    done = False
    while not done:
        train_loader.batch_sampler.set_epoch(epoch)
        for micro, batch in enumerate(train_loader):
            batch = adapter.to_device(batch)
            with _autocast(cfg.train.precision):
                loss = adapter.loss(batch) / cfg.train.grad_accum
            scaler.scale(loss).backward()
            running += loss.item()
            if (micro + 1) % cfg.train.grad_accum:
                continue
            scaler.unscale_(opt)
            gnorm = torch.nn.utils.clip_grad_norm_(adapter.trainable_parameters(), cfg.train.max_grad_norm)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1

            if step % cfg.train.log_every == 0:
                row = {"step": step, "epoch": epoch, "train/loss": running / cfg.train.log_every,
                       "train/grad_norm": float(gnorm), "train/lr": sched.get_last_lr()[0],
                       "train/elapsed_min": (time.time() - t0) / 60,
                       "train/gpu_mem_gb": torch.cuda.max_memory_allocated() / 1e9}
                log.info(" ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items()))
                log_row(row)
                running = 0.0

            if step % cfg.train.eval_every == 0 or step == cfg.train.max_steps:
                m = evaluate(adapter, dev_loader, cfg)
                log.info("step %d dev: %s", step, m)
                log_row({"step": step, **{f"dev/{k}": v for k, v in m.items()}})
                if m["wer"] < best_wer:
                    best_wer, bad_evals = m["wer"], 0
                    if cfg.train.save_best:
                        adapter.save(out / "best")
                        (out / "best" / "dev_metrics.json").write_text(json.dumps({"step": step, **m}, indent=2))
                else:
                    bad_evals += 1
                    if cfg.train.early_stopping_patience and bad_evals >= cfg.train.early_stopping_patience:
                        log.info("early stopping at step %d (best dev WER %.2f)", step, best_wer)
                        done = True
                        break
            if step >= cfg.train.max_steps:
                done = True
                break
        epoch += 1

    adapter.save(out / "last")
    tb.close()
    summary = {"best_dev_wer": best_wer, "steps": step, "epochs": epoch,
               "train_minutes": (time.time() - t0) / 60}
    (out / "train_summary.json").write_text(json.dumps(summary, indent=2))
    return summary


Writing /content/bodhan-asr-marathi/src/bodhan_asr/trainer.py


In [15]:
%%writefile /content/bodhan-asr-marathi/scripts/train.py
"""Fine-tune Indic-Transcribe on Marathi.

    python scripts/train.py --config configs/mr_full.yaml [train.max_steps=500 ...]
"""

from __future__ import annotations

import argparse
import json
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from bodhan_asr.config import ExperimentConfig  # noqa: E402
from bodhan_asr.trainer import train  # noqa: E402


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("overrides", nargs="*", help="section.key=value")
    args = ap.parse_args()
    cfg = ExperimentConfig.load(args.config, args.overrides)
    Path(cfg.train.output_dir).mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
        handlers=[logging.StreamHandler(), logging.FileHandler(Path(cfg.train.output_dir) / "train.log")],
    )
    print(json.dumps(train(cfg), indent=2))


if __name__ == "__main__":
    main()


Writing /content/bodhan-asr-marathi/scripts/train.py


In [16]:
%%writefile /content/bodhan-asr-marathi/scripts/evaluate.py
"""Score a checkpoint (or the untouched base model) on a manifest.

    # zero-shot baseline
    python scripts/evaluate.py --config configs/mr_full.yaml --split test --tag base
    # fine-tuned
    python scripts/evaluate.py --config configs/mr_full.yaml --split test \
        --checkpoint /content/runs/mr_full/best --tag finetuned
"""

from __future__ import annotations

import argparse
import json
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from bodhan_asr.config import ExperimentConfig  # noqa: E402
from bodhan_asr.data import ManifestDataset  # noqa: E402
from bodhan_asr.model import AsrAdapter  # noqa: E402
from bodhan_asr.trainer import evaluate, make_loader  # noqa: E402


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--split", choices=["dev", "test"], default="test")
    ap.add_argument("--checkpoint", default=None, help="dir saved by AsrAdapter.save")
    ap.add_argument("--tag", required=True)
    ap.add_argument("--out_dir", default=None)
    ap.add_argument("overrides", nargs="*")
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

    cfg = ExperimentConfig.load(args.config, args.overrides)
    adapter = AsrAdapter.from_config(cfg, checkpoint=args.checkpoint)
    manifest = cfg.data.test_manifest if args.split == "test" else cfg.data.dev_manifest
    loader = make_loader(ManifestDataset(manifest), adapter, cfg, shuffle=False)

    out = Path(args.out_dir or Path(cfg.train.output_dir) / "eval")
    metrics = evaluate(adapter, loader, cfg, predictions_path=out / f"{args.split}_{args.tag}.jsonl")
    metrics.update(split=args.split, tag=args.tag, checkpoint=args.checkpoint)
    (out / f"{args.split}_{args.tag}_metrics.json").write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()


Writing /content/bodhan-asr-marathi/scripts/evaluate.py


In [17]:
%%writefile /content/bodhan-asr-marathi/configs/mr_full.yaml
# Full fine-tuning of Indic-Transcribe-Core on SPRING-INX Marathi (A100 80GB).
data:
  train_manifest: /content/data/prepared/train.jsonl
  dev_manifest: /content/data/prepared/dev.jsonl
  test_manifest: /content/data/prepared/test.jsonl
  lang: mr
  max_batch_seconds: 600    # padded audio-seconds per step (~40-50 utts)
  max_batch_size: 48
  num_workers: 8

model:
  name_or_path: /content/models/indic-transcribe-core
  strategy: full            # full | decoder_only | lora
  freeze_encoder_layers: 0
  prompt_mode: mixed        # corpus writes English loanwords in Latin script
  spec_augment: true
  label_smoothing: 0.0

train:
  output_dir: /content/runs/mr_full
  seed: 42
  lr: 1.0e-5                # low LR: model already knows Marathi; avoid forgetting
  encoder_lr_scale: 1.0
  weight_decay: 0.01
  warmup_steps: 150
  max_steps: 1500
  grad_accum: 1
  max_grad_norm: 1.0
  precision: bf16
  eval_every: 250
  log_every: 10
  early_stopping_patience: 3
  num_beams: 1
  max_new_tokens: 256


Writing /content/bodhan-asr-marathi/configs/mr_full.yaml


In [18]:
%%writefile /content/bodhan-asr-marathi/configs/mr_lora.yaml
# Parameter-efficient variant: LoRA on all encoder+decoder attention projections.
data:
  train_manifest: /content/data/prepared/train.jsonl
  dev_manifest: /content/data/prepared/dev.jsonl
  test_manifest: /content/data/prepared/test.jsonl
  lang: mr
  max_batch_seconds: 600
  max_batch_size: 48
  num_workers: 8

model:
  name_or_path: /content/models/indic-transcribe-core
  strategy: lora
  lora_r: 32
  lora_alpha: 64
  lora_dropout: 0.05
  prompt_mode: mixed
  spec_augment: true

train:
  output_dir: /content/runs/mr_lora
  lr: 2.0e-4                # LoRA needs a much higher LR than full FT
  warmup_steps: 150
  max_steps: 1500
  precision: bf16
  eval_every: 250
  early_stopping_patience: 3


Writing /content/bodhan-asr-marathi/configs/mr_lora.yaml


In [19]:
%%writefile /content/bodhan-asr-marathi/scripts/smoke_test.py
"""Fast end-to-end sanity check before spending GPU hours:
loads the model, decodes a few dev clips, and runs one optimisation step."""

from __future__ import annotations

import io
import logging
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import pyarrow.parquet as pq  # noqa: E402
import soundfile as sf  # noqa: E402
import torch  # noqa: E402
import transformers  # noqa: E402

from bodhan_asr.config import ExperimentConfig  # noqa: E402
from bodhan_asr.data import ManifestDataset  # noqa: E402
from bodhan_asr.model import AsrAdapter  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
cfg = ExperimentConfig.load(sys.argv[1] if len(sys.argv) > 1 else "configs/mr_full.yaml")
print("torch", torch.__version__, "| transformers", transformers.__version__)

# 1. Native sample rate of the corpus (decides whether resampling matters at all).
raw = next(Path("/content/data/raw/data").glob("test-*.parquet"))
row = pq.read_table(raw, columns=["audio"]).slice(0, 1).to_pylist()[0]
info = sf.info(io.BytesIO(row["audio"]["bytes"]))
print("source audio:", info.samplerate, "Hz,", info.channels, "ch,", info.subtype)

# 2. Load + decode a handful of dev utterances.
adapter = AsrAdapter.from_config(cfg)
print("prompt ids:", adapter.prompt)
ds = ManifestDataset(cfg.data.dev_manifest, max_items=6)
batch = adapter.to_device(adapter.collate([ds[i] for i in range(len(ds))]))
adapter.model.eval()
with torch.autocast("cuda", dtype=torch.bfloat16):
    t = time.time()
    hyps = adapter.transcribe(batch)
    print(f"decode {len(hyps)} utts in {time.time() - t:.1f}s")
    for r, h in zip(batch["texts"], hyps):
        print("REF:", r, "\nHYP:", h, "\n")
    print("eval loss:", adapter.loss(batch).item())

# 3. One training step (checks grads flow and memory is sane).
adapter.train_mode()
opt = torch.optim.AdamW(adapter.trainable_parameters(), lr=1e-6)
with torch.autocast("cuda", dtype=torch.bfloat16):
    loss = adapter.loss(batch)
loss.backward()
gn = torch.nn.utils.clip_grad_norm_(adapter.trainable_parameters(), 1.0)
opt.step()
print(f"train loss {loss.item():.4f} grad_norm {gn:.3f} peak_mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print("SMOKE TEST OK")


Writing /content/bodhan-asr-marathi/scripts/smoke_test.py


## 3. Data preparation

In [20]:
# 3. Prepare data: parquet -> 16 kHz FLAC + NeMo-style JSONL manifests (train / dev / test)
%cd /content/bodhan-asr-marathi
!python scripts/prepare_data.py --raw_dir /content/data/raw --out_dir /content/data/prepared --train_shards 10 --dev_items 1000 2>&1 | grep -v Warning
!wc -l /content/data/prepared/*.jsonl; head -c 600 /content/data/prepared/dev.jsonl

/content/bodhan-asr-marathi
2026-09-23 15:33:12,490 NumExpr defaulting to 12 threads.
2026-09-23 15:35:46,889 train: {'kept': 16260, 'too_long': 14, 'chars_per_sec': 33, 'empty_text': 3} | 45.26 h kept
2026-09-23 15:35:56,607 dev: {'kept': 999, 'chars_per_sec': 1} | 2.90 h kept
2026-09-23 15:36:14,011 test: {'kept': 1929, 'empty_text': 1} | 5.05 h kept
    999 /content/data/prepared/dev.jsonl
   1929 /content/data/prepared/test.jsonl
  16260 /content/data/prepared/train.jsonl
  19188 total
{"audio_filepath": "/content/data/prepared/audio/dev/utt00008561.flac", "duration": 2.47, "text": "नमस्कार, मी अनुप मटके.", "source_lang": "mr", "target_lang": "mr", "pnc": "yes", "utt_id": "utt00008561"}
{"audio_filepath": "/content/data/prepared/audio/dev/utt00009252.flac", "duration": 12.29, "text": "तिने सन्मानित करण्यात आले होते, ज्यामूळे त्यांना न्युयाॅर्क येथे कोलंबिया विश्वविद्यालयात उच्चप

## 4. Smoke test (decode real audio + one optimiser step before spending GPU hours)

In [21]:
# 4. Smoke test: load model, decode a few dev clips, one optimisation step
%cd /content/bodhan-asr-marathi
!mkdir -p /content/logs && python scripts/smoke_test.py configs/mr_full.yaml 2>&1 | grep -v "Warning" | tee /content/logs/smoke.txt

/content/bodhan-asr-marathi
2026-09-23 15:36:23,593 NumExpr defaulting to 12 threads.
torch 2.11.0+cu128 | transformers 5.16.1
source audio: 16000 Hz, 1 ch, PCM_16
Loading weights: 100%|██████████| 1923/1923 [00:00<00:00, 4072.68it/s]
2026-09-23 15:36:35,214 strategy=full trainable params 1221.4M / 1221.4M (100.0%)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
prompt ids: [7, 4, 18, 134, 134, 5, 8, 11, 13, 15]
decode 6 utts in 2.9s
REF: नमस्कार, मी अनुप मटके. 
HYP: नमस्कार मी अनुप मस्के 

REF: तिने सन्मानित करण्यात आले होते, ज्यामूळे त्यांना न्युयाॅर्क येथे कोलंबिया विश्वविद्यालयात उच्चपदवी प्राप्त करण्याची संधी मिळाली. आपल्या 
HYP: तिने सन्मानित करण्यात आले होते ज्यामुळे त्यांना न्यूयॉर्क येथे कोलंबिया विश्वविद्यालयात उच्च पदवी प्राप्त करण्याची संधी मिळाली आपल्या 

REF

## 5. Full fine-tuning (background process)

Note: this cell was re-run once by accident, which launched a second identical run into the same directory. It was allowed to finish and used as a reproducibility check (test WER 35.90 vs 35.76). The trainer now refuses to reuse a run directory.

In [22]:
# 5. Full fine-tuning run (background process; logs -> /content/runs/mr_full/train.log, TensorBoard -> tb/)
%cd /content/bodhan-asr-marathi
!mkdir -p /content/runs/mr_full && nohup python scripts/train.py --config configs/mr_full.yaml > /content/runs/mr_full/stdout.txt 2>&1 &
!sleep 5; tail -3 /content/runs/mr_full/stdout.txt

/content/bodhan-asr-marathi
  "epochs": 5,
  "train_minutes": 21.114652983347575
}


## 6. Zero-shot baseline on the official test split (native and mixed prompt modes)

In [23]:
# 6. Zero-shot baseline on the official test split, in both output modes (runs alongside training)
%cd /content/bodhan-asr-marathi
!mkdir -p /content/runs/baseline && nohup sh -c "python scripts/evaluate.py --config configs/mr_full.yaml --split test --tag base_mixed --out_dir /content/runs/baseline model.prompt_mode=mixed data.num_workers=4 && python scripts/evaluate.py --config configs/mr_full.yaml --split test --tag base_native --out_dir /content/runs/baseline model.prompt_mode=native data.num_workers=4" > /content/runs/baseline/stdout.txt 2>&1 &

/content/bodhan-asr-marathi


## 7. Persist to Google Drive, pull the final code, evaluate the fine-tuned model

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
%cd /content
!git clone -q https://github.com/deveshh01/bodhan-asr-marathi /content/bodhan-asr-marathi && ls /content/bodhan-asr-marathi

/content
configs  README.md  requirements.txt  scripts  src


In [37]:
!grep "dev:" /content/runs/mr_full/train.log | tail -7
!cd /content/bodhan-asr-marathi && python scripts/evaluate.py --config configs/mr_full.yaml --split test --tag finetuned --checkpoint /content/runs/mr_full/best 2>&1 | tail -9

2026-09-23 16:09:57,141 INFO step 0 (zero-shot) dev: {'wer': 38.344095710066576, 'cer': 19.315520083282728, 'n': 999, 'loss': 1.4910249347272126}
2026-09-23 16:13:33,569 INFO step 250 dev: {'wer': 34.03914501676928, 'cer': 16.203695671033227, 'n': 999, 'loss': 0.7789817944816921}
2026-09-23 16:17:51,138 INFO step 500 dev: {'wer': 33.68373629674125, 'cer': 15.784679448251932, 'n': 999, 'loss': 0.7437073769776718}
2026-09-23 16:21:24,530 INFO step 750 dev: {'wer': 32.95289583020474, 'cer': 15.312743992365748, 'n': 999, 'loss': 0.7335997716240261}
2026-09-23 16:24:57,242 INFO step 1000 dev: {'wer': 32.847774941182365, 'cer': 15.236401492148868, 'n': 999, 'loss': 0.7269570931144382}
2026-09-23 16:28:28,533 INFO step 1250 dev: {'wer': 32.677579216098515, 'cer': 15.236401492148868, 'n': 999, 'loss': 0.725024295889813}
2026-09-23 16:31:59,014 INFO step 1500 dev: {'wer': 32.742654052159985, 'cer': 15.271970157022643, 'n': 999, 'loss': 0.7241167322449062}
{
  "wer": 35.76084712267351,
  "cer": 

In [38]:
!rsync -a --exclude last/ /content/runs/mr_full /content/runs/baseline /content/drive/MyDrive/bodhan-asr-marathi/runs/ && du -sh /content/drive/MyDrive/bodhan-asr-marathi/runs/*

2.2M	/content/drive/MyDrive/bodhan-asr-marathi/runs/baseline
2.0K	/content/drive/MyDrive/bodhan-asr-marathi/runs/lora_eval.txt
6.0K	/content/drive/MyDrive/bodhan-asr-marathi/runs/lora_stdout.txt
2.3G	/content/drive/MyDrive/bodhan-asr-marathi/runs/mr_full
8.0K	/content/drive/MyDrive/bodhan-asr-marathi/runs/mr_full.stdout
11K	/content/drive/MyDrive/bodhan-asr-marathi/runs/mr_lora


## 8. LoRA comparison (r=32, all attention projections, 1.7% of parameters)

Colab's preinstalled `torchao 0.10` makes recent `peft` refuse to import; it was uninstalled first (`pip uninstall -y torchao`).

In [40]:
%cd /content
!rm -rf /content/runs/mr_lora && cd /content/bodhan-asr-marathi && nohup sh -c "python scripts/train.py --config configs/mr_lora.yaml > /content/runs/lora_stdout.txt 2>&1; python scripts/evaluate.py --config configs/mr_lora.yaml --split test --tag finetuned_lora --checkpoint /content/runs/mr_lora/best > /content/runs/lora_eval.txt 2>&1; rsync -a --exclude last/ /content/runs/ /content/drive/MyDrive/bodhan-asr-marathi/runs/" > /dev/null 2>&1 &

/content


In [42]:
!grep -E "dev:|early" /content/runs/lora_stdout.txt | tail -8; tail -9 /content/runs/lora_eval.txt

2026-09-23 16:40:03,974 INFO step 0 (zero-shot) dev: {'wer': 38.344095710066576, 'cer': 19.315520083282728, 'n': 999, 'loss': 1.4910249295442}
2026-09-23 16:44:55,365 INFO step 250 dev: {'wer': 34.25439255143415, 'cer': 16.257482432549665, 'n': 999, 'loss': 0.7820883859758792}
2026-09-23 16:49:30,973 INFO step 500 dev: {'wer': 33.17815487810983, 'cer': 15.48364708944218, 'n': 999, 'loss': 0.7520696054334226}
2026-09-23 16:54:05,516 INFO step 750 dev: {'wer': 33.03799369274666, 'cer': 15.363060640235968, 'n': 999, 'loss': 0.7397574704626332}
2026-09-23 16:58:40,776 INFO step 1000 dev: {'wer': 32.69760224257897, 'cer': 15.226858679621758, 'n': 999, 'loss': 0.7333773011746614}
2026-09-23 17:03:16,403 INFO step 1250 dev: {'wer': 32.817740401461684, 'cer': 15.278042855903532, 'n': 999, 'loss': 0.7297321609828783}
2026-09-23 17:07:49,980 INFO step 1500 dev: {'wer': 32.807728888221455, 'cer': 15.283248026372863, 'n': 999, 'loss': 0.7303357875865438}
{
  "wer": 35.89947792230776,
  "cer": 17.0

## 9. Report: training curves, results table, largest per-utterance changes

In [43]:
%cd /content/bodhan-asr-marathi
!python scripts/report.py --runs /content/runs/mr_full /content/runs/mr_lora --baseline /content/runs/baseline --out /content/drive/MyDrive/bodhan-asr-marathi/docs && ls -la /content/drive/MyDrive/bodhan-asr-marathi/docs

/content/bodhan-asr-marathi
## Results

| model | split | WER % | CER % | utts |
|---|---|---|---|---|
| base_mixed | test | 41.19 | 20.85 | 1929 |
| base_native | test | 41.33 | 20.88 | 1929 |
| finetuned | test | 35.76 | 16.94 | 1929 |
| finetuned_lora | test | 35.90 | 17.08 | 1929 |

## Largest changes on test (baseline -> fine-tuned)

**utt00098796** (utt WER 650% -> 25%)

- REF: TELCECIGNOUNIOSNITTR आणि IIMB Platform
- BASE: टी ई एल सी ई सी आय जी एन ओ यू एन आय ओ ओ एस एन आय टी टी आर आणि आय आय एम बी प्लॅटफॉर्म
- FT: TELCECIGNOUNIOSNITTR आणि IMB platform

**utt00060600** (utt WER 500% -> 0%)

- REF: काहून
- BASE: कागद दल म्हणून काही काऊ
- FT: काहून

**utt00055284** (utt WER 400% -> 200%)

- REF: हा
- BASE: हां हां जुडे भाई
- FT: हा जुडे भाई

**utt00043637** (utt WER 200% -> 0%)

- REF: हम्म
- BASE: ठीक आहे
- FT: हम्म

**utt00078709** (utt WER 88% -> 412%)

- REF: हा ते ही miss नाही केलं, सगळे बोलतं होते म्हटलं काय boring असेल काय बघू ह्यांना
- BASE: हां ओपियो उसको कोण मिस कर सकता है 